# 05 — Batch Script & HPC Job Tutorial

This notebook shows how to run `metacountregressor` searches in batch mode:
from a local parallel shell loop to a full PBS/SLURM cluster job array.

**Topics covered:**
1. Write a reusable `run_experiment.py` script
2. Run multiple seeds locally (parallel shell loop)
3. Collect and compare results from many runs
4. Submit PBS/SLURM job arrays
5. Walltime-aware search (auto-stop before scheduler kills the job)
6. Useful helper commands for monitoring jobs

**Previous:** [04_linear_speed_prediction.ipynb](04_linear_speed_prediction.ipynb)

---

```python
from metacountregressor import get_help
get_help('batch')   # print the full inline batch guide
```

In [ ]:
from metacountregressor import get_help

# Print the complete inline batch / HPC guide
get_help('batch')

## Step 1 — Write `run_experiment.py`

This script is your **reusable batch entry point**.  It accepts all key parameters
from the command line so you can vary algorithm, seed, R, and iteration count
without editing the script.

Save the cell below as `run_experiment.py` in your working directory:

In [ ]:
# ── run_experiment.py template ────────────────────────────────────────────────
# Save this to run_experiment.py and run with:
#   python run_experiment.py sa 42 200 2000

SCRIPT = '''
"""run_experiment.py — reusable metacountregressor batch entry point."""
import sys
import numpy as np
from metacountregressor import (
    ExperimentBuilder,
    ModelConstraints,
    SearchOutputConfig,
    load_example16_3_model_data,
)

# ── Command-line arguments ────────────────────────────────────────────────────
# Usage: python run_experiment.py <algo> <seed> <R> <max_iter> [max_time_sec]
# Defaults: sa  42  200  2000  0
ALGO     = sys.argv[1] if len(sys.argv) > 1 else 'sa'
SEED     = int(sys.argv[2]) if len(sys.argv) > 2 else 42
R        = int(sys.argv[3]) if len(sys.argv) > 3 else 200
MAX_ITER = int(sys.argv[4]) if len(sys.argv) > 4 else 2000
MAX_TIME = float(sys.argv[5]) if len(sys.argv) > 5 else 0   # 0 = no limit

print(f"Starting: algo={ALGO}  seed={SEED}  R={R}  max_iter={MAX_ITER}  max_time={MAX_TIME}s")

# ── Load data ─────────────────────────────────────────────────────────────────
df = load_example16_3_model_data()
# df = pd.read_csv('your_data.csv')   # ← replace with your own data if needed

# ── Build constraints ─────────────────────────────────────────────────────────
c = (
    ModelConstraints()
    .force_include('OFFSET')
    .no_zi('LENGTH', 'CURVES', 'WIDTH', 'SLOPE')
    .no_random('URB')
    .allow_random('CURVES', distributions=['lognormal'])
)

# ── Build evaluator ───────────────────────────────────────────────────────────
builder = ExperimentBuilder(df, id_col='ID', y_col='FREQ', offset_col='OFFSET')

evaluator = builder.build_evaluator(
    variables=['AADT', 'LENGTH', 'SPEED', 'CURVES', 'URB', 'AVEPRE'],
    constraints=c,
    default_roles=[0, 1, 2, 3, 5],
    max_latent_classes=1,
    R=R,
)

# ── Run search ────────────────────────────────────────────────────────────────
result = builder.run(
    evaluator,
    algo=ALGO,
    max_iter=MAX_ITER,
    seed=SEED,
    max_time=MAX_TIME if MAX_TIME > 0 else None,
    output_config=SearchOutputConfig(
        output_dir='results',
        experiment_name=f'nb_search_{ALGO}_seed{SEED}',
        search_description=f'{ALGO} search seed={SEED} R={R} max_iter={MAX_ITER}',
    ),
)

print(f"Done.  BIC={result.best_score:.3f}  iter={result.iteration}  "
      f"time={result.elapsed_time:.1f}s  saved={result.saved_to}")
'''

# Write the script to disk
import pathlib
script_path = pathlib.Path('run_experiment.py')
script_path.write_text(SCRIPT.strip())
print(f'Written: {script_path.resolve()}')

## Step 2 — Test the script locally

In [ ]:
# Run a single quick test from inside the notebook
import subprocess, sys

proc = subprocess.run(
    [sys.executable, 'run_experiment.py', 'sa', '42', '200', '30'],
    capture_output=True, text=True
)

print('STDOUT:')
print(proc.stdout)
if proc.returncode != 0:
    print('STDERR:')
    print(proc.stderr)

## Step 3 — Parallel local runs (multiple seeds)

Run the same algorithm with 5 different seeds in parallel using a shell background loop:

In [ ]:
# Generate the shell commands for a parallel seed sweep
algo     = 'sa'
seeds    = [1, 2, 3, 4, 5]
R        = 200
max_iter = 500

print('# Run these commands in a terminal (Unix/Mac/WSL/Git Bash):')
print()
for seed in seeds:
    print(f'python run_experiment.py {algo} {seed} {R} {max_iter} &')
print('wait')
print('echo "All runs complete"')
print()
print('# On Windows PowerShell:')
for seed in seeds:
    print(f'Start-Process python -ArgumentList "run_experiment.py {algo} {seed} {R} {max_iter}"')
print()

In [ ]:
# ── Or run them sequentially from Python ──────────────────────────────────────
# (remove the comment from each line to run)

import subprocess, sys

seeds_to_run = [42, 7, 13]   # quick 3-seed sweep

print('Running 3-seed sweep (max_iter=30 for demo) ...')
print()

for seed in seeds_to_run:
    r = subprocess.run(
        [sys.executable, 'run_experiment.py', 'sa', str(seed), '200', '30'],
        capture_output=True, text=True
    )
    # Extract the final Done line
    for line in r.stdout.strip().splitlines():
        if line.startswith('Done') or line.startswith('Starting'):
            print(f'  seed={seed}: {line}')

print()
print('Sweep complete.')

## Step 4 — Collect results from all runs

In [ ]:
import json
import pathlib
import pandas as pd

results_dir = pathlib.Path('results')

if not results_dir.exists():
    print('results/ directory does not exist yet. Run some searches first.')
else:
    records = []
    for f in sorted(results_dir.glob('*.json')):
        try:
            with open(f) as fh:
                data = json.load(fh)
            records.append({
                'file':        f.name,
                'experiment':  data.get('experiment_name', ''),
                'algorithm':   data.get('algorithm', ''),
                'best_bic':    data.get('best_score',  float('inf')),
                'iterations':  data.get('iteration',   0),
                'elapsed_s':   data.get('elapsed_time', 0),
            })
        except Exception as e:
            print(f'  Skipping {f.name}: {e}')

    if records:
        summary = pd.DataFrame(records).sort_values('best_bic')
        print(f'Found {len(summary)} result files in results/')
        print()
        print(summary.to_string(index=False))
        print()
        print(f'Best BIC overall : {summary["best_bic"].min():.3f}')
        print(f'Best algorithm   : {summary.loc[summary["best_bic"].idxmin(), "algorithm"]}')
    else:
        print('No JSON result files found yet.')

## Step 4b — Turn results tables into notebook and markdown slides

Any fitted-model or search summary table can be rendered directly in the notebook and then written to markdown for Marp slides.

In the quickstart notebook, `builder.print_coefficients(fit)` gives the fitted-model coefficient table. In this batch tutorial we already have `summary`, so we use that same export pattern here.

In [ ]:
from pathlib import Path

if "summary" not in locals() or summary.empty:
    raise RuntimeError("Run Step 4 first so the `summary` table is available.")

slide_table = summary.copy()
numeric_cols = slide_table.select_dtypes(include="number").columns
slide_table[numeric_cols] = slide_table[numeric_cols].round(3)

slide_dir = Path("results/slide_assets")
slide_dir.mkdir(parents=True, exist_ok=True)

table_md = slide_table.to_markdown(index=False)
table_html = slide_table.to_html(index=False)

(slide_dir / "batch_results_table.md").write_text(table_md, encoding="utf-8")
(slide_dir / "batch_results_table.html").write_text(table_html, encoding="utf-8")

slide_deck = f"""---
marp: true
title: Batch Search Results
paginate: true
---

# Batch Search Results

## Ranked runs

{table_md}
"""

(slide_dir / "batch_results_slides.md").write_text(slide_deck.strip(), encoding="utf-8")

print(f"Saved slide assets to: {slide_dir.resolve()}")
print("If you already have a fitted model, swap `slide_table` for `builder.print_coefficients(fit)` to export coefficient tables the same way.")
slide_table

## Step 5 — Algorithm comparison sweep

Run all three algorithms with the same seed to compare convergence:

In [ ]:
# Generate a 3-algorithm comparison script
print('# Run all three algorithms with seed=42:')
print()
for algo in ['sa', 'de', 'hs']:
    print(f'python run_experiment.py {algo} 42 200 500 &')
print('wait')
print()
print('# Then collect results and compare BIC:')
print('# python -c "import json,pathlib; ...(see Step 4 above)"')

## Step 6 — PBS/SLURM job scripts

The following cells generate ready-to-use HPC job scripts.
Edit the `#PBS` / `#SBATCH` parameters for your cluster.

In [ ]:
PBS_SCRIPT = '''
#!/bin/bash
#PBS -N metacount_sa
#PBS -l nodes=1:ppn=4
#PBS -l walltime=04:00:00
#PBS -l mem=16gb
#PBS -j oe
#PBS -o logs/job_${PBS_JOBID}.log
#PBS -t 1-10            # array: jobs 1 through 10 (each gets a different seed)

# ── Environment setup ─────────────────────────────────────────────────────────
module load python/3.11
cd $PBS_O_WORKDIR
source venv/bin/activate

# ── Unique seed from array index ──────────────────────────────────────────────
SEED=$PBS_ARRAYID         # 1, 2, ..., 10
ALGO="sa"
R=200
MAX_ITER=99999            # effectively unlimited — walltime stops the search

# PBS_WALLTIME is automatically detected by metacountregressor
# The search will stop cleanly before the walltime expires.

echo "Starting: SEED=$SEED  ALGO=$ALGO  WALLTIME=$PBS_WALLTIME"
python run_experiment.py $ALGO $SEED $R $MAX_ITER
echo "Job complete: SEED=$SEED"
'''

import pathlib
pathlib.Path('job_pbs.sh').write_text(PBS_SCRIPT.strip())
print('PBS job script written to: job_pbs.sh')
print()
print('Submit with:  qsub job_pbs.sh')
print('Check status: qstat -u $USER')
print('Cancel job:   qdel <jobid>')

In [ ]:
SLURM_SCRIPT = '''
#!/bin/bash
#SBATCH --job-name=metacount_sa
#SBATCH --nodes=1
#SBATCH --ntasks-per-node=4
#SBATCH --time=04:00:00
#SBATCH --mem=16G
#SBATCH --output=logs/%x_%A_%a.log
#SBATCH --array=1-10        # array: jobs 1 through 10

# ── Environment setup ─────────────────────────────────────────────────────────
module load python/3.11
cd $SLURM_SUBMIT_DIR
source venv/bin/activate

# ── Unique seed from array index ──────────────────────────────────────────────
SEED=$SLURM_ARRAY_TASK_ID   # 1, 2, ..., 10
ALGO="sa"
R=200
MAX_ITER=99999               # effectively unlimited

# SLURM_TIME_LIMIT is automatically detected by metacountregressor.
# The search will stop cleanly before the time limit expires.

echo "Starting: SEED=$SEED  ALGO=$ALGO  TIMELIMIT=$SLURM_TIME_LIMIT"
python run_experiment.py $ALGO $SEED $R $MAX_ITER
echo "Job complete: SEED=$SEED"
'''

pathlib.Path('job_slurm.sh').write_text(SLURM_SCRIPT.strip())
print('SLURM job script written to: job_slurm.sh')
print()
print('Submit with:  sbatch job_slurm.sh')
print('Check status: squeue -u $USER')
print('Cancel job:   scancel <jobid>')

## Step 7 — Walltime-aware search

On PBS/SLURM, `metacountregressor` reads the scheduler walltime automatically.
On a local machine, you can set it manually:

In [ ]:
# Demo: manually set max_time to 30 seconds
import numpy as np
from metacountregressor import (
    ExperimentBuilder, ModelConstraints, SearchOutputConfig,
    load_example16_3_model_data,
)

df = load_example16_3_model_data()

c = (
    ModelConstraints()
    .force_include('OFFSET')
    .no_zi('LENGTH', 'CURVES')
    .no_random('URB')
)

builder = ExperimentBuilder(df, id_col='ID', y_col='FREQ', offset_col='OFFSET')

evaluator = builder.build_evaluator(
    variables=['AADT', 'LENGTH', 'SPEED', 'CURVES', 'URB'],
    constraints=c,
    default_roles=[0, 1, 2, 3],
    max_latent_classes=1,
    R=200,
)

# max_time=30 means: stop after 30 seconds even if max_iter not reached
result = builder.run(
    evaluator,
    algo='sa',
    max_iter=99999,     # virtually unlimited — time stops it
    max_time=30,        # stop after 30 seconds
    seed=42,
)

print('─' * 60)
print(f'Search stopped after : {result.elapsed_time:.1f} s  (max_time=30)')
print(f'Iterations completed : {result.iteration}')
print(f'Best BIC             : {result.best_score:.3f}')
print('─' * 60)
print()
print('On PBS/SLURM, walltime is detected automatically — no max_time needed.')
print('The package reads PBS_WALLTIME or SLURM_TIME_LIMIT from the environment.')

## Step 8 — Useful monitoring commands

The cells below print handy commands for checking job status, reading log files,
and collecting results after the batch completes.

In [ ]:
print("""
USEFUL COMMANDS FOR BATCH JOB MANAGEMENT
=========================================

PBS / TORQUE
  qstat -u $USER           # list your running jobs
  qstat -f <jobid>         # detailed job info
  qdel <jobid>             # cancel a job
  tail -f logs/job_*.log   # stream the latest log file

SLURM
  squeue -u $USER          # list your running jobs
  scontrol show job <id>   # detailed job info
  scancel <jobid>          # cancel a job
  tail -f logs/*.log       # stream the latest log

CHECK RESULTS (run after all jobs finish)
  python -c "
import json, pathlib, pandas as pd
rows = []
for f in pathlib.Path('results').glob('*.json'):
    d = json.load(open(f))
    rows.append({'file': f.name, 'bic': d.get('best_score'), 'algo': d.get('algorithm')})
df = pd.DataFrame(rows).sort_values('bic')
print(df.to_string(index=False))
print('Best BIC:', df['bic'].min())
  "

COUNT COMPLETED RESULTS
  ls results/*.json | wc -l   # how many JSON results exist

QUICK STATUS CHECK (shows last line of each log)
  tail -1 logs/*.log
""")

## Step 9 — Advanced: multi-algorithm, multi-LC sweep

Generate a full factorial sweep script: 3 algorithms × 5 seeds × 2 LC settings = 30 jobs:

In [ ]:
# Generate all combinations
import itertools

algos       = ['sa', 'de', 'hs']
seeds       = [1, 2, 3, 4, 5]
max_lc_list = [1, 2]
R           = 200
max_iter    = 1000

lines = ['#!/bin/bash', '']
count = 0
for algo, seed, max_lc in itertools.product(algos, seeds, max_lc_list):
    # Pass max_lc as extra arg — you'd need to update run_experiment.py to accept it
    lines.append(
        f'python run_experiment.py {algo} {seed} {R} {max_iter} 0 {max_lc} &'
    )
    count += 1

lines += ['', 'wait', f'echo "{count} jobs complete"']

sweep_script = '\n'.join(lines)
pathlib.Path('run_sweep.sh').write_text(sweep_script)

print(f'Sweep script written to: run_sweep.sh')
print(f'Total jobs: {count}')
print()
print('First 10 lines:')
print('\n'.join(lines[:12]))

## Summary: batch workflow checklist

```
[ ] 1. Write run_experiment.py  (see Step 1)
[ ] 2. Test locally with a short run  (python run_experiment.py sa 42 200 30)
[ ] 3. Create results/ and logs/ directories
[ ] 4. Choose your sweep parameters (algorithms, seeds, R, LC)
[ ] 5. Write the PBS/SLURM job script  (see Step 6)
[ ] 6. Submit:  qsub job_pbs.sh  OR  sbatch job_slurm.sh
[ ] 7. Monitor:  qstat -u $USER  OR  squeue -u $USER
[ ] 8. Collect:  python collect_results.py  (see Step 4)
[ ] 9. Load the best spec and do a final high-R re-fit
```

```python
from metacountregressor import get_help
get_help('batch')          # inline batch guide
get_help('metaheuristics') # algorithm comparison and parameters
```

---

You have now completed all five notebooks:

| Notebook | What you learned |
| --- | --- |
| [00_quickstart](00_quickstart.ipynb) | Install, load, first run |
| [01_crash_frequency_search](01_crash_frequency_search.ipynb) | Mixed NB search, constraints |
| [02_latent_class_fc_validation](02_latent_class_fc_validation.ipynb) | LC model, class probabilities, FC validation |
| [03_cmf_aadt_search](03_cmf_aadt_search.ipynb) | CMF model, two-component structure |
| [04_linear_speed_prediction](04_linear_speed_prediction.ipynb) | Gaussian linear, duration models |
| **05 (this one)** | Batch scripts, PBS/SLURM job arrays, walltime |

**Full API reference:** `README.md` in the package root.